In [ ]:
# ============================================================
# Caso de estudio: Caja de recaudo con tope de seguridad
# Empresa: PagaYa S.A.S. | Asignatura: Programación 1
# Base de datos: archivos CSV (se pueden abrir y editar en Colab)
# ============================================================

import csv
import os

# ------------------------------------------------------------
# Configuración de archivos CSV (la "base de datos")
# ------------------------------------------------------------
archivoTurnos = "turnos.csv"
archivoTransacciones = "transacciones.csv"

camposTurnos = ["idTurno", "cajero", "tope", "numTransacciones",
                "recaudoTotal", "promedio", "motivoCierre"]
camposTransacciones = ["idTransaccion", "idTurno", "monto"]

mensajeSuspension = "CAJA SUSPENDIDA: se alcanzó el tope de recaudo. Diríjase a tesorería."


# ============================================================
# FUNCIONES DE ACCESO A CSV
# ============================================================

def crearArchivoSiNoExiste(rutaArchivo, campos):
    # Crea el CSV con su encabezado si todavía no existe
    if not os.path.exists(rutaArchivo):
        with open(rutaArchivo, "w", newline="", encoding="utf-8") as archivo:
            escritor = csv.DictWriter(archivo, fieldnames=campos)
            escritor.writeheader()


def leerCsv(rutaArchivo, campos):
    # Devuelve el contenido del CSV como lista de diccionarios
    crearArchivoSiNoExiste(rutaArchivo, campos)
    with open(rutaArchivo, "r", newline="", encoding="utf-8") as archivo:
        return list(csv.DictReader(archivo))


def escribirCsv(rutaArchivo, campos, filas):
    # Sobrescribe el CSV completo con la lista de filas recibida
    with open(rutaArchivo, "w", newline="", encoding="utf-8") as archivo:
        escritor = csv.DictWriter(archivo, fieldnames=campos)
        escritor.writeheader()
        escritor.writerows(filas)


def siguienteId(filas, nombreCampo):
    # Calcula el próximo id autoincremental
    if len(filas) == 0:
        return 1
    return max(int(fila[nombreCampo]) for fila in filas) + 1


# ============================================================
# FUNCIONES DE FORMATO Y VALIDACIÓN
# ============================================================

def formatearMiles(valor):
    # 1050000 -> "1.050.000"
    return f"{int(valor):,}".replace(",", ".")


def formatearDecimales(valor):
    # 350000 -> "350.000,00" (miles con punto, decimales con coma)
    texto = f"{float(valor):,.2f}"
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")


def leerEnteroPositivo(mensaje):
    # RF1 / RF3: pide un valor hasta que sea entero y mayor que cero
    while True:
        textoIngresado = input(mensaje).strip()
        try:
            numero = int(textoIngresado)
            if numero > 0:
                return numero
            print("  Valor rechazado: debe ser mayor que cero.")
        except ValueError:
            print("  Valor rechazado: debe ser un número entero.")


def calcularPromedio(recaudoTotal, numTransacciones):
    # RF6: si no hubo transacciones, el promedio es 0 (evita división por cero)
    if numTransacciones == 0:
        return 0
    return recaudoTotal / numTransacciones


# ============================================================
# REPORTE (RF6)
# ============================================================

def mostrarReporte(turno):
    # Imprime el reporte de un turno con el formato solicitado
    print("\n===== REPORTE DE TURNO =====")
    print(f"Cajero:                {turno['cajero']}")
    print(f"Tope asignado:         ${formatearMiles(turno['tope'])}")
    print(f"Transacciones:         {turno['numTransacciones']}")
    print(f"Recaudo total:         ${formatearMiles(turno['recaudoTotal'])}")
    print(f"Promedio/transacción:  ${formatearDecimales(turno['promedio'])}")
    print(f"Motivo de cierre:      {turno['motivoCierre']}")
    print("============================\n")


# ============================================================
# CREATE: iniciar un turno y atender clientes (RF1 - RF6)
# ============================================================

def iniciarTurno():
    # RF1: datos de inicio de turno
    cajero = input("Nombre o código del cajero: ").strip()
    while cajero == "":
        cajero = input("  El cajero no puede estar vacío. Intente de nuevo: ").strip()
    tope = leerEnteroPositivo("Tope máximo de recaudo (COP): ")

    recaudoTotal = 0
    listaMontos = []
    motivoCierre = ""

    # RF2: atender clientes mientras la caja esté activa
    while True:
        textoMonto = input(f"Cliente {len(listaMontos) + 1} - Monto recibido (0 o FIN para cerrar): ").strip()

        # RF5: cierre por fin de cola
        if textoMonto == "0" or textoMonto.upper() == "FIN":
            motivoCierre = "Fin de cola"
            break

        # RF3: validación de montos
        try:
            monto = int(textoMonto)
        except ValueError:
            print("  Monto rechazado: valor no numérico.")
            continue
        if monto <= 0:
            print("  Monto rechazado: debe ser mayor que cero.")
            continue

        # Monto válido: se cuenta y se acumula
        listaMontos.append(monto)
        recaudoTotal += monto
        print(f"  Acumulado: ${formatearMiles(recaudoTotal)}")

        # RF4: suspensión por tope (la transacción que cruza el tope SÍ se registra)
        if recaudoTotal >= tope:
            print("\n" + mensajeSuspension)
            motivoCierre = "Tope alcanzado"
            break

    # Guardar el turno en el CSV
    turnos = leerCsv(archivoTurnos, camposTurnos)
    transacciones = leerCsv(archivoTransacciones, camposTransacciones)

    idTurno = siguienteId(turnos, "idTurno")
    numTransacciones = len(listaMontos)
    promedio = calcularPromedio(recaudoTotal, numTransacciones)

    turnoNuevo = {
        "idTurno": idTurno,
        "cajero": cajero,
        "tope": tope,
        "numTransacciones": numTransacciones,
        "recaudoTotal": recaudoTotal,
        "promedio": promedio,
        "motivoCierre": motivoCierre,
    }
    turnos.append(turnoNuevo)

    idTransaccion = siguienteId(transacciones, "idTransaccion")
    for monto in listaMontos:
        transacciones.append({"idTransaccion": idTransaccion,
                              "idTurno": idTurno,
                              "monto": monto})
        idTransaccion += 1

    escribirCsv(archivoTurnos, camposTurnos, turnos)
    escribirCsv(archivoTransacciones, camposTransacciones, transacciones)

    # RF6: reporte final
    mostrarReporte(turnoNuevo)


# ============================================================
# READ: consultar turnos y transacciones
# ============================================================

def listarTurnos():
    # Muestra todos los turnos guardados
    turnos = leerCsv(archivoTurnos, camposTurnos)
    if len(turnos) == 0:
        print("\nNo hay turnos registrados.\n")
        return
    print("\n--- TURNOS REGISTRADOS ---")
    for turno in turnos:
        print(f"ID {turno['idTurno']} | Cajero: {turno['cajero']} | "
              f"Tope: ${formatearMiles(turno['tope'])} | "
              f"Trans: {turno['numTransacciones']} | "
              f"Recaudo: ${formatearMiles(turno['recaudoTotal'])} | "
              f"Cierre: {turno['motivoCierre']}")
    print()


def verReporteTurno():
    # Busca un turno por id y muestra su reporte
    idBuscado = input("ID del turno a consultar: ").strip()
    turnos = leerCsv(archivoTurnos, camposTurnos)
    for turno in turnos:
        if turno["idTurno"] == idBuscado:
            mostrarReporte(turno)
            return
    print("  No se encontró un turno con ese ID.\n")


def listarTransacciones():
    # Muestra las transacciones de un turno
    idBuscado = input("ID del turno: ").strip()
    transacciones = leerCsv(archivoTransacciones, camposTransacciones)
    encontradas = [t for t in transacciones if t["idTurno"] == idBuscado]
    if len(encontradas) == 0:
        print("  Ese turno no tiene transacciones (o no existe).\n")
        return
    print(f"\n--- TRANSACCIONES DEL TURNO {idBuscado} ---")
    for transaccion in encontradas:
        print(f"Transacción {transaccion['idTransaccion']}: "
              f"${formatearMiles(transaccion['monto'])}")
    print()


# ============================================================
# Función auxiliar: recalcular un turno tras editar/eliminar
# ============================================================

def recalcularTurno(idTurno):
    # Recalcula número de transacciones, recaudo y promedio de un turno
    turnos = leerCsv(archivoTurnos, camposTurnos)
    transacciones = leerCsv(archivoTransacciones, camposTransacciones)

    montosTurno = [int(t["monto"]) for t in transacciones if t["idTurno"] == idTurno]
    recaudoTotal = sum(montosTurno)
    numTransacciones = len(montosTurno)

    for turno in turnos:
        if turno["idTurno"] == idTurno:
            turno["numTransacciones"] = numTransacciones
            turno["recaudoTotal"] = recaudoTotal
            turno["promedio"] = calcularPromedio(recaudoTotal, numTransacciones)
    escribirCsv(archivoTurnos, camposTurnos, turnos)


# ============================================================
# UPDATE: modificar turno o transacción
# ============================================================

def actualizarTurno():
    # Permite cambiar el cajero o el tope de un turno
    idBuscado = input("ID del turno a actualizar: ").strip()
    turnos = leerCsv(archivoTurnos, camposTurnos)

    for turno in turnos:
        if turno["idTurno"] == idBuscado:
            nuevoCajero = input(f"Nuevo cajero [{turno['cajero']}] (Enter para mantener): ").strip()
            if nuevoCajero != "":
                turno["cajero"] = nuevoCajero

            textoTope = input(f"Nuevo tope [{turno['tope']}] (Enter para mantener): ").strip()
            if textoTope != "":
                try:
                    nuevoTope = int(textoTope)
                    if nuevoTope > 0:
                        turno["tope"] = nuevoTope
                    else:
                        print("  Tope rechazado (debe ser mayor que cero). Se mantiene el anterior.")
                except ValueError:
                    print("  Tope rechazado (no numérico). Se mantiene el anterior.")

            escribirCsv(archivoTurnos, camposTurnos, turnos)
            print("  Turno actualizado.\n")
            return
    print("  No se encontró un turno con ese ID.\n")


def actualizarTransaccion():
    # Modifica el monto de una transacción y recalcula el turno
    idBuscado = input("ID de la transacción a modificar: ").strip()
    transacciones = leerCsv(archivoTransacciones, camposTransacciones)

    for transaccion in transacciones:
        if transaccion["idTransaccion"] == idBuscado:
            nuevoMonto = leerEnteroPositivo(f"Nuevo monto [{transaccion['monto']}]: ")
            transaccion["monto"] = nuevoMonto
            escribirCsv(archivoTransacciones, camposTransacciones, transacciones)
            recalcularTurno(transaccion["idTurno"])
            print("  Transacción actualizada y turno recalculado.\n")
            return
    print("  No se encontró una transacción con ese ID.\n")


# ============================================================
# DELETE: eliminar turno o transacción
# ============================================================

def eliminarTurno():
    # Elimina un turno junto con todas sus transacciones
    idBuscado = input("ID del turno a eliminar: ").strip()
    turnos = leerCsv(archivoTurnos, camposTurnos)
    transacciones = leerCsv(archivoTransacciones, camposTransacciones)

    turnosRestantes = [t for t in turnos if t["idTurno"] != idBuscado]
    if len(turnosRestantes) == len(turnos):
        print("  No se encontró un turno con ese ID.\n")
        return

    transaccionesRestantes = [t for t in transacciones if t["idTurno"] != idBuscado]
    escribirCsv(archivoTurnos, camposTurnos, turnosRestantes)
    escribirCsv(archivoTransacciones, camposTransacciones, transaccionesRestantes)
    print("  Turno y sus transacciones eliminados.\n")


def eliminarTransaccion():
    # Elimina una transacción y recalcula el turno al que pertenecía
    idBuscado = input("ID de la transacción a eliminar: ").strip()
    transacciones = leerCsv(archivoTransacciones, camposTransacciones)

    idTurnoAfectado = None
    transaccionesRestantes = []
    for transaccion in transacciones:
        if transaccion["idTransaccion"] == idBuscado:
            idTurnoAfectado = transaccion["idTurno"]
        else:
            transaccionesRestantes.append(transaccion)

    if idTurnoAfectado is None:
        print("  No se encontró una transacción con ese ID.\n")
        return

    escribirCsv(archivoTransacciones, camposTransacciones, transaccionesRestantes)
    recalcularTurno(idTurnoAfectado)
    print("  Transacción eliminada y turno recalculado.\n")


# ============================================================
# MENÚ PRINCIPAL
# ============================================================

def mostrarMenu():
    print("========== PAGAYA S.A.S. - CAJA DE RECAUDO ==========")
    print("1. Iniciar turno (CREATE)")
    print("2. Listar turnos (READ)")
    print("3. Ver reporte de un turno (READ)")
    print("4. Ver transacciones de un turno (READ)")
    print("5. Actualizar turno (UPDATE)")
    print("6. Actualizar transacción (UPDATE)")
    print("7. Eliminar turno (DELETE)")
    print("8. Eliminar transacción (DELETE)")
    print("0. Salir")


def main():
    # Crea los CSV si no existen y ejecuta el menú
    crearArchivoSiNoExiste(archivoTurnos, camposTurnos)
    crearArchivoSiNoExiste(archivoTransacciones, camposTransacciones)

    while True:
        mostrarMenu()
        opcion = input("Seleccione una opción: ").strip()

        if opcion == "1":
            iniciarTurno()
        elif opcion == "2":
            listarTurnos()
        elif opcion == "3":
            verReporteTurno()
        elif opcion == "4":
            listarTransacciones()
        elif opcion == "5":
            actualizarTurno()
        elif opcion == "6":
            actualizarTransaccion()
        elif opcion == "7":
            eliminarTurno()
        elif opcion == "8":
            eliminarTransaccion()
        elif opcion == "0":
            print("Hasta luego.")
            break
        else:
            print("  Opción no válida.\n")


main()

========== PAGAYA S.A.S. - CAJA DE RECAUDO ==========
1. Iniciar turno (CREATE)
2. Listar turnos (READ)
3. Ver reporte de un turno (READ)
4. Ver transacciones de un turno (READ)
5. Actualizar turno (UPDATE)
6. Actualizar transacción (UPDATE)
7. Eliminar turno (DELETE)
8. Eliminar transacción (DELETE)
0. Salir
Seleccione una opción: 1
Nombre o código del cajero: 01
Tope máximo de recaudo (COP): 1000000
Cliente 1 - Monto recibido (0 o FIN para cerrar): 500000
  Acumulado: $500.000
Cliente 2 - Monto recibido (0 o FIN para cerrar): 200000
  Acumulado: $700.000
Cliente 3 - Monto recibido (0 o FIN para cerrar): 301000
  Acumulado: $1.001.000

CAJA SUSPENDIDA: se alcanzó el tope de recaudo. Diríjase a tesorería.

===== REPORTE DE TURNO =====
Cajero:                01
Tope asignado:         $1.000.000
Transacciones:         3
Recaudo total:         $1.001.000
Promedio/transacción:  $333.666,67
Motivo de cierre:      Tope alcanzado

========== PAGAYA S.A.S. - CAJA DE RECAUDO ==========
1. Inici